# A retrieval pipeline: chunk, embed, search, answer with citations

Machine Learning Engineer track, lesson MLE9. Part of [datafromzero](https://datafromzero.github.io), data and text CC BY 4.0. The text is the site's own: eight lessons of the SQL stage, also CC BY 4.0.

**Goals:** cut documents into chunks and see the size trade-off, embed the chunks and search them, give the best chunks to a language model and get an answer with citations, and make the system refuse when nothing relevant is found.

**How to use this notebook:** run the cells from top to bottom. When in doubt, **Run all from the top**: restart the session and run every cell from the first.
Every exercise has a cell for your code and a check cell that compares your answer with a known answer. Hints and solutions are folded away
under each exercise, free to open. The working cell of Exercise 2 starts empty: you write it. No key and no account are needed: both models are open and run in this notebook. A free CPU runtime is enough (the language model is about 3.3 GB to download, and the cell of unit 3 takes about 3 minutes on two CPU threads); a GPU runtime may be faster, but it was not timed.

## Setup

In [ ]:
import importlib.metadata
import math
import numbers
import re
import statistics
import subprocess
import sys
import time
import urllib.request
from typing import Literal

for pin in ["transformers==5.24.1", "sentence-transformers==6.3.0", "pydantic==2.14.4"]:                                                         # the versions this notebook was run with
    package, version = pin.split("==")
    try:
        installed = importlib.metadata.version(package)
    except importlib.metadata.PackageNotFoundError:
        installed = None
    if installed != version:                                                  # install the pinned version when Colab has another
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pin], check=True)

import numpy as np
import torch
from pydantic import BaseModel, ValidationError
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

SEED = 2026       # the seed of the whole track
torch.set_num_threads(2)       # two threads, the size of a free Colab CPU

EXERCISES = ['Exercise 1', 'Exercise 2', 'Exercise 3']


# Run this cell as it is. It also holds the answer checker used by the exercise cells.
# Run all from the top: restart the session, run every cell from the first, then read the last cell.

RESULTS = {}


def same(got, expected, tol=1e-9):
    """True when two values agree: floats within `tol`, lists and dicts item by item."""
    try:
        if isinstance(expected, dict):
            return isinstance(got, dict) and got.keys() == expected.keys() and all(same(got[k], expected[k], tol) for k in expected)
        if isinstance(expected, (list, tuple)):
            return isinstance(got, (list, tuple)) and len(got) == len(expected) and all(same(g, e, tol) for g, e in zip(got, expected))
        if isinstance(expected, bool) or isinstance(got, bool):
            return type(got) is type(expected) and got == expected
        if isinstance(expected, float):
            return isinstance(got, numbers.Real) and math.isclose(got, expected, rel_tol=1e-9, abs_tol=tol)
        if isinstance(expected, int):
            return isinstance(got, numbers.Integral) and got == expected
        return type(got) is type(expected) and got == expected
    except Exception:
        return False


def check(label, name, expected, traps=None, tol=1e-9):
    """Compare the variable called `name` with the known answer. It never stops Run all."""
    got = globals().get(name)
    if got is None:
        RESULTS[label] = None
        print(f"{label}: give {name} a value, then run this cell again.")
        return False
    if same(got, expected, tol):
        RESULTS[label] = True
        print(f"{label}: correct.")
        return True
    RESULTS[label] = False
    print(f"{label}: not yet. Expected {expected!r}, but {name} is {got!r}.")
    for wrong, message in (traps.items() if isinstance(traps, dict) else (traps or [])):
        if same(got, wrong, tol):
            print("Hint:", message)
    return False

## The corpus and the questions

Eight lessons of the SQL stage, read from the release tag `v1.0.0` (or from this repository's copy of the same files), as in lesson MLE7: `CORPUS` holds each lesson's paragraphs as (section heading, text), without front matter, figures, code, tables and exercise sections. Run this cell as it is.

`DEV` is 16 questions written for this lesson, two for each lesson. Each holds the question, the lesson that answers it, phrases of the paragraph that holds the answer (a chunk that contains one of them has the answer) and patterns that a correct answer must match. `TRAPS` are 8 questions that the eight lessons do not answer: four far from SQL and four that use SQL words. Run both cells as they are.

In [ ]:
# The corpus: eight published lessons of the SQL stage (CC BY 4.0), read from the release tag v1.0.0, or from this repository's own copy of the same files.
TEXT_BASE = "https://raw.githubusercontent.com/datafromzero/datafromzero.github.io/v1.0.0/src/content/docs/sql/"
TEXT_LOCAL = "../../src/content/docs/sql/"      # the same files, when this notebook runs inside a clone of the repository
LESSONS = ["null-handling", "group-by-having", "inner-join", "left-join-anti-join", "window-functions", "ctes", "dates-and-time", "case-expressions"]
SKIP_SECTIONS = {"Warm-up", "Finish it", "On your own", "Check yourself"}      # recall prompts and exercises are not teaching text


def read_lesson(slug):
    """The raw text of one lesson file: downloaded from the tag, or the local copy."""
    try:
        with urllib.request.urlopen(TEXT_BASE + slug + ".mdx", timeout=30) as reply:
            return reply.read().decode("utf-8")
    except Exception:
        with open(TEXT_LOCAL + slug + ".mdx", encoding="utf-8") as handle:
            return handle.read()


def paragraphs(raw):
    """The prose paragraphs of a lesson as (section heading, text): no front matter, figures, code, tables or exercise sections."""
    raw = re.sub(r"\A---\n.*?\n---\n", "", raw, flags=re.S)
    raw = re.sub(r"<figure.*?</figure>|<svg.*?</svg>|<RunSQL.*?/>|```.*?```", "", raw, flags=re.S)
    found, section = [], ""
    for block in re.split(r"\n\s*\n", raw):
        block = block.strip()
        if block.startswith("#"):
            section = block.lstrip("#").strip()
        elif block and section not in SKIP_SECTIONS and not block.startswith(("<", "|", "{/*")):
            block = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", re.sub(r"<[^>]+>", "", block)).replace("*", "").replace("`", "")
            block = re.sub(r"\s+", " ", block).strip()
            if len(block.split()) >= 8:                                           # a few words are a label, not a paragraph
                found.append((section, block))
    return found


CORPUS = {slug: paragraphs(read_lesson(slug)) for slug in LESSONS}
paras = [(slug, text) for slug in LESSONS for _, text in CORPUS[slug]]            # (lesson, paragraph) for every paragraph
texts, labels = [text for _, text in paras], [slug for slug, _ in paras]
print(len(paras), "paragraphs,", sum(len(text.split()) for text in texts), "words,", len(LESSONS), "lessons")

In [ ]:
DEV = [
    ('Why does WHERE customer_id = NULL never return a row?', 'null-handling', ['= NULL is NULL for every row', 'test it with IS NULL, never with'], ['IS NULL']),
    ('Which function replaces a missing value with a default?', 'null-handling', ['COALESCE(x, d) returns the first'], ['COALESCE']),
    ('What is the difference between WHERE and HAVING?', 'group-by-having', ['HAVING runs after the groups are built', 'WHERE runs before any group exists', 'They run at different moments'], ['before', 'after']),
    ('How many orders does store 2 have in all?', 'group-by-having', ['store 2 has 716'], ['716']),
    ('What happens to a row that has no partner in an inner join?', 'inner-join', ['Only paired rows are returned', 'drops the rows that found no pair', 'A row with no partner, or with an empty key, is missing'], ['drop|missing|not (be )?(returned|included|kept)|disappear|removed|excluded|left out|no row']),
    ('If a key repeats on both sides, how many rows does a join return?', 'inner-join', ['gives a × b rows'], ['a ?[×x*] ?b|product|multipl']),
    ('How do I find customers who never placed an order?', 'left-join-anti-join', ['To list the rows of one table that have no partner', 'ANTI JOIN gives the left rows with no match'], ['LEFT JOIN|ANTI JOIN']),
    ('Why did my LEFT JOIN lose the unmatched rows after I added a WHERE on the right table?', 'left-join-anti-join', ['On an unpaired row o.status is NULL', 'ON decides which rows pair', 'so the row leaves'], ['\\bON\\b']),
    ('How can I add a total beside every row without losing rows?', 'window-functions', ['A window function keeps every row'], ['OVER|window']),
    ('Can I filter on the result of a window function in WHERE?', 'window-functions', ['Windows run after WHERE and HAVING'], ["cannot|can't|CTE|not"]),
    ('What is a CTE?', 'ctes', ['A CTE (common table expression) gives one step of a query a name'], ['name']),
    ('Does a CTE stay in the database after the query ends?', 'ctes', ['It lives for one statement', 'The name lasts for this one statement'], ['\\bno\\b|not|only|one statement']),
    ('How do I get the first day of the month of a timestamp?', 'dates-and-time', ["date_trunc('month', x) cuts a timestamp back to the start of its month"], ['date_trunc']),
    ('Why does BETWEEN miss orders placed on the last day of a month?', 'dates-and-time', ['A date written alone means the start of that day', 'the BETWEEN version is short by the 9 orders'], ['00:00|start of (that|the) day|midnight|time']),
    ('What does CASE return when no WHEN is true and there is no ELSE?', 'case-expressions', ['a row that matches no WHEN gets NULL'], ['NULL']),
    ('Which WHEN wins when two tests are both true?', 'case-expressions', ['stops at the first one that is true', 'The first true WHEN wins'], ['first']),
]
TRAPS = [
    ('How do I schedule a pipeline to run every night?', 'far'),
    ('What does a p-value of 0.03 mean?', 'far'),
    ('How do I install pandas?', 'far'),
    ('What is the capital of Australia?', 'far'),
    ('How do I rank rows with RANK()?', 'near'),
    ('What is a self join?', 'near'),
    ('How do I create an index on a column?', 'near'),
    ('What is the difference between UNION and UNION ALL?', 'near'),
]
QUESTIONS = [item[0] for item in DEV]
print(len(DEV), "questions,", len(TRAPS), "traps")

## The two models

The small encoder of lesson MLE7, `all-MiniLM-L6-v2` (Apache 2.0), which embeds the chunks and the questions. Then `granite-4.0-1b` (1.63 billion numbers), the open model of lesson MLE8 (made by IBM, Apache 2.0, downloaded without an account), and its `chat` function. Each is pinned to one revision of its model repository. Run both cells as they are: the second one downloads the model once and loads it on the GPU when there is one, on the CPU otherwise.

In [ ]:
EMB_NAME = "sentence-transformers/all-MiniLM-L6-v2"
EMB_REVISION = "1110a243fdf4706b3f48f1d95db1a4f5529b4d41"                    # the exact version of the model files
emb = SentenceTransformer(EMB_NAME, revision=EMB_REVISION, device="cpu")
print(sum(p.numel() for p in emb.parameters()), emb.max_seq_length)

In [ ]:
LLM_NAME = "ibm-granite/granite-4.0-1b"
LLM_REVISION = "6a7381ba1f54d684ff508d991aeb7dc580157103"                    # the exact version of the model files
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(LLM_NAME, revision=LLM_REVISION)
llm = AutoModelForCausalLM.from_pretrained(LLM_NAME, revision=LLM_REVISION, dtype=torch.float32).to(DEVICE).eval()
END_MARKS = {tok.eos_token_id}                                              # the end-of-text marker: <|end_of_text|>


def chat(messages, max_new_tokens=100, temperature=0.0, seed=None):
    """One request. temperature 0 picks the most likely token every time; above 0 it samples."""
    prompt = tok.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt", return_dict=True)["input_ids"].to(DEVICE)
    settings = {"do_sample": False, "repetition_penalty": 1.0}
    if temperature > 0:
        settings = {"do_sample": True, "temperature": temperature, "top_p": 1.0, "top_k": 0, "repetition_penalty": 1.0}
        torch.manual_seed(SEED if seed is None else seed)
    start = time.perf_counter()
    with torch.no_grad():
        out = llm.generate(input_ids=prompt, attention_mask=torch.ones_like(prompt), max_new_tokens=max_new_tokens, **settings)
    seconds = time.perf_counter() - start
    written = out[0][prompt.shape[1]:].tolist()
    return {"text": tok.decode(written, skip_special_tokens=True), "prompt_tokens": prompt.shape[1],
            "completion_tokens": sum(1 for t in written if t not in END_MARKS), "seconds": seconds}


print(sum(p.numel() for p in llm.parameters()), DEVICE)

## Chunks and the search helper

`make_chunks` joins whole paragraphs of one lesson into chunks of at most `target_words` words (a paragraph longer than that stays whole), and keeps the lesson and the section of each chunk. `hits` counts the questions for which a chunk among the `k` most similar to the question holds the phrase of the answer. Run this cell as it is.

In [ ]:
def make_chunks(target_words):
    """Chunks of whole paragraphs, each (lesson, section, text), at most `target_words` words unless one paragraph is longer."""
    chunks = []
    for slug in LESSONS:
        current, count, head = [], 0, ""
        for section, text in CORPUS[slug]:
            size = len(text.split())
            if current and count + size > target_words:
                chunks.append((slug, head, " ".join(current)))
                current, count = [], 0
            if not current:
                head = section
            current.append(text)
            count += size
        if current:
            chunks.append((slug, head, " ".join(current)))
    return chunks


def hits(chunks, vectors, k=3):
    """How many of the 16 questions have a chunk with the phrase of the answer among the k most similar chunks."""
    scores = emb.encode(QUESTIONS) @ vectors.T
    found = 0
    for row, (_, _, phrases, _) in zip(scores, DEV):
        top = np.argsort(-row)[:k]
        found += any(phrase in chunks[i][2] for i in top for phrase in phrases)
    return found

## Unit 1: Chunk size

**Predict first.** The same eight lessons are cut into chunks of about 40, 120 and 400 words. For each size the cell prints the number of chunks, the mean length of a chunk in tokens, how many chunks are longer than the 256 tokens the encoder reads, how many of the 16 questions have the phrase of the answer in their top 3 chunks, and the mean tokens of those 3 chunks together. Which size finds the most answers, and what does it cost?

In [ ]:
for target in [40, 120, 400]:
    chunks = make_chunks(target)
    vectors = emb.encode([chunk[2] for chunk in chunks])
    sizes = [len(emb.tokenizer.tokenize(chunk[2])) for chunk in chunks]
    top3 = np.argsort(-(emb.encode(QUESTIONS) @ vectors.T), axis=1)[:, :3]
    context = statistics.mean(sum(sizes[i] for i in row) for row in top3)
    print(target, len(chunks), round(statistics.mean(sizes)), sum(size > emb.max_seq_length for size in sizes), hits(chunks, vectors), round(context))

## Unit 2: Embed and search

**Predict first.** With chunks of 120 words, the question below is embedded and compared with every chunk. What is the cosine of its best chunk, and from which lesson? Over all 16 questions, how often is the best chunk from the lesson that answers, and how often does one of the top 3 hold the phrase of the answer?

In [ ]:
CHUNKS = make_chunks(120)
VECTORS = emb.encode([chunk[2] for chunk in CHUNKS])                          # one row of 384 numbers for each chunk, computed once


def search(question, k=3):
    """The k chunks most similar to the question, as (cosine, number of the chunk)."""
    scores = VECTORS @ emb.encode([question])[0]
    return [(round(float(scores[i]), 3), int(i)) for i in np.argsort(-scores)[:k]]


for score, i in search("How do I find customers who never placed an order?"):
    print(score, CHUNKS[i][0], "|", CHUNKS[i][1], "|", CHUNKS[i][2][:60])
first_right = sum(CHUNKS[search(question, 1)[0][1]][0] == lesson for question, lesson, _, _ in DEV)
print(first_right, hits(CHUNKS, VECTORS))

## Unit 3: Answer from the chunks, with citations

**Predict first.** The prompt gives the 3 best chunks as numbered sources and asks for JSON with an answer and the numbers of the sources it used. It is run on one question for each lesson, 8 in all. How many of the 8 replies will be valid JSON whose cited numbers are among the sources shown, and how many will contain the words that a correct answer must match?

In [ ]:
class Cited(BaseModel):
    answer: str | None                                                        # null when the sources do not hold the answer
    sources: list[int]


SYSTEM = ("Answer the question using only the numbered sources. Reply with JSON only, like {\"answer\": \"one or two sentences\", \"sources\": [1]}, "
          "where sources lists the numbers of the sources you used. If the sources do not contain the answer, reply {\"answer\": null, \"sources\": []}.")


def answer(question, k=3):
    """Search, build the prompt from the k best chunks, ask the model, and check what comes back."""
    found = search(question, k)
    sources = "\n".join(f"[{n}] {CHUNKS[i][2]}" for n, (score, i) in enumerate(found, start=1))
    reply = chat([{"role": "system", "content": SYSTEM}, {"role": "user", "content": f"Sources:\n{sources}\n\nQuestion: {question}"}], max_new_tokens=100)
    try:
        parsed = Cited.model_validate_json(reply["text"])
        ok = all(1 <= n <= k for n in parsed.sources)                         # every cited number is one of the numbers shown
        return {**reply, "found": found, "answer": parsed.answer, "sources": parsed.sources, "valid": ok}
    except ValidationError:
        return {**reply, "found": found, "answer": None, "sources": [], "valid": False}


chosen = list(range(0, 16, 2))                                                # the first question of each lesson
results = [answer(DEV[i][0]) for i in chosen]


def matches(i, result):
    """True when the answer holds the words that a correct answer must match."""
    return result["answer"] is not None and all(re.search(pattern, result["answer"], re.I) for pattern in DEV[i][3])


for i, r in zip(chosen, results):
    print(r["valid"], r["sources"], matches(i, r))
print(sum(r["valid"] for r in results), sum(matches(i, r) for i, r in zip(chosen, results)))
for i, r in zip(chosen, results):
    if not matches(i, r):
        print(DEV[i][0], "->", r["answer"])

## Unit 4: Abstain when nothing is relevant

**Predict first.** A question whose best chunk is a poor match should get no answer at all. The threshold is the lowest best-chunk cosine among the 16 questions, rounded down to 0.05: none of these 16 is refused. At that threshold, how many of the 8 trap questions are refused before the model is called, and which ones get through?

In [ ]:
best_dev = [search(question, 1)[0][0] for question in QUESTIONS]
best_trap = [(search(question, 1)[0][0], question, kind) for question, kind in TRAPS]
THRESHOLD = math.floor(min(best_dev) * 20) / 20                               # the lowest cosine of a real question, rounded down to 0.05
print(THRESHOLD, round(min(best_dev), 3), round(max(best_dev), 3))
for score, question, kind in sorted(best_trap):
    print(score, kind, score < THRESHOLD, question)


def answer_or_abstain(question, k=3):
    """Refuse without calling the model when even the best chunk is a poor match."""
    if search(question, 1)[0][0] < THRESHOLD:
        return {"answer": None, "sources": [], "abstained": True}
    return {**answer(question, k), "abstained": False}


print(sum(score < THRESHOLD for score, _, _ in best_trap), "of", len(best_trap), "traps refused")

## Finish it

### Exercise 1: Finish it

**Task:** **A smaller chunk.** Cut the corpus into chunks of about 80 words with `make_chunks(80)` and embed them as in unit 1. Put two numbers in the list `answer_1`: the number of chunks, and how many of the 16 questions have the phrase of the answer among their top 3 chunks.

**Expected:** `answer_1` is a list of two whole numbers.

In [ ]:
chunks_80 = make_chunks(80)
vectors_80 = emb.encode([chunk[2] for chunk in chunks_80])                    # given
answer_1 = None  # replace None with [number of chunks, hits for the top 3 chunks]

In [ ]:
_ = check("Exercise 1", "answer_1", [124, 12])

<details>
<summary>Hint</summary>

`hits(chunks, vectors)` takes the chunks and their vectors, and its default is the top 3.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_1 = [len(chunks_80), hits(..., ...)]
```

</details>

<details>
<summary>Solution</summary>

```python
chunks_80 = make_chunks(80)
vectors_80 = emb.encode([chunk[2] for chunk in chunks_80])
answer_1 = [len(chunks_80), hits(chunks_80, vectors_80)]
```

[124, 12]: chunks of about 80 words give 124 chunks, and 12 of the 16 questions find the phrase of their answer in the top 3, as many as with the chunks of 120 words in unit 1.

</details>

## On your own

### Exercise 2: On your own

**Task:** **How many answers within k.** Before you start, say in one sentence what you expect: as k grows from 1 to 10, the hits can only stay or rise. Why? Then write the cell yourself, in the empty cell below: with the chunks of 120 words (`CHUNKS` and `VECTORS`), put the number of the 16 questions that have the phrase of the answer among their top 1, top 5 and top 10 chunks in the list `answer_2`.

**Expected:** `answer_2` is a list of three whole numbers from 0 to 16, not decreasing.

In [ ]:
_ = check("Exercise 2", "answer_2", [9, 12, 13])

<details>
<summary>Hint</summary>

`hits` has a third argument, `k`. A list comprehension over `(1, 5, 10)` gives the three numbers.

</details>

<details>
<summary>Half-solution</summary>

```python
answer_2 = [hits(CHUNKS, VECTORS, k) for k in ...]
```

</details>

<details>
<summary>Solution</summary>

```python
answer_2 = [hits(CHUNKS, VECTORS, k) for k in (1, 5, 10)]
```

[9, 12, 13]: the phrase is in the top chunk for 9 questions, in the top 5 for 12 and in the top 10 for 13. The count can only stay or rise as k grows, because the top 5 contain the top 1. Every extra chunk is more tokens in the prompt.

</details>

## Check yourself

### Exercise 3: Check yourself

**Task:** **Spot the flaw.** A colleague reports that the citations are right for every answer of unit 3, because every cited number is one of the numbers of the sources shown. Change the count so that an answer counts only when at least one source it cites holds a phrase of the answer (`DEV[i][2]`), and put the number of such answers in `answer_3`.

**Expected:** `answer_3` is a whole number from 0 to 8.

In [ ]:
cited_ok = sum(r["valid"] and len(r["sources"]) > 0 for r in results)                # colleague: "every cited number is one of the sources shown"
answer_3 = cited_ok
# change the count: an answer counts only when a source it cites holds a phrase of the answer

In [ ]:
_ = check("Exercise 3", "answer_3", 6, traps={8: "That only checks that the cited numbers exist. Look inside the cited chunks."})

<details>
<summary>Hint</summary>

For each answer, `r["found"][n - 1]` is the (cosine, chunk number) of the source the model calls `n`, and `CHUNKS[chunk number][2]` is its text.

</details>

<details>
<summary>Half-solution</summary>

```python
supported = 0
for i, r in zip(chosen, results):
    cited_text = " ".join(CHUNKS[r["found"][n - 1][1]][2] for n in r["sources"] if 1 <= n <= len(r["found"]))
    ...
```

</details>

<details>
<summary>Solution</summary>

```python
supported = 0
for i, r in zip(chosen, results):
    cited_text = " ".join(CHUNKS[r["found"][n - 1][1]][2] for n in r["sources"] if 1 <= n <= len(r["found"]))
    supported += any(phrase in cited_text for phrase in DEV[i][2])
answer_3 = supported
```

The answer is 6. All 8 answers cite numbers that exist, which is what the flawed count measures, but only 6 of them cite a source that holds the phrase of the answer. A valid citation is a pointer; whether it leads to the answer is a second check, which is what the new count reads: the text of the cited source.

</details>

## Run all from the top

Restart the session, run every cell from the first, then read this cell last.

In [ ]:
for label in EXERCISES:
    state = RESULTS.get(label)
    print(f"{label}: " + ("correct" if state else "not yet" if state is False else "not started"))
print(f"{sum(1 for label in EXERCISES if RESULTS.get(label))} of {len(EXERCISES)} exercises correct.")